# Entity linking: model walkthrough

Follow two mentions through detection, candidate retrieval and scoring. Models use random weights; gold BIO labels fix the example spans. XLM-R configuration and tokenizer files are downloaded or reused from cache.


## Complete flow: function → input → output

| Function | Input | Output |
|---|---|---|
| `tokenizer.encode_text` | Input text | Token IDs, attention mask, offsets |
| `mention_detection` | Text, tokens, offsets | Mention vectors, texts, spans |
| `entity_typing` | Mention vectors | Type probabilities |
| `retriever` | Mention texts | Candidate IDs, records, types, priors |
| `tokenizer.encode_descriptions` | Candidate names + descriptions, mention/candidate counts | Token IDs and masks grouped by mention/candidate |
| `description_encoder` | Description IDs + masks | Candidate vectors |
| `description_matching` | Mention + candidate vectors, validity mask | Description probabilities |
| `entity_disambiguation` | Type probabilities, known types, priors, description probabilities, validity mask | Candidate scores + NIL |
| `argmax` | Scores | Candidate ID, or `None` for NIL |

Keep mention order throughout. Return each mention's text, span and selected ID. This untrained demo supplies gold `bio_labels` to bypass detection predictions.


In [ ]:
import pandas as pd
import torch
from torch import nn
from transformers import AutoTokenizer, XLMRobertaConfig, XLMRobertaModel
from IPython.display import display

torch.manual_seed(7)
torch.set_num_threads(2)

class LinkingTokenizer:
    """Share one vocabulary across mention and description inputs."""

    def __init__(self, backbone_name):
        self.tokenizer = AutoTokenizer.from_pretrained(backbone_name, use_fast=True)

    def encode_text(self, text):
        """Return model inputs and character offsets for one untruncated text."""
        tokens = self.tokenizer(text, return_offsets_mapping=True, return_tensors="pt")
        return {
            "token_ids": tokens["input_ids"],
            "mask_ids": tokens["attention_mask"],
            "offsets": tokens["offset_mapping"][0],
        }

    def encode_descriptions(self, texts, num_mentions, num_candidates, max_length=32):
        """Pad/truncate descriptions and group tokens as [mentions, candidates, tokens]."""
        if num_mentions < 1 or num_candidates < 1 or len(texts) != num_mentions * num_candidates:
            raise ValueError("Provide one description per candidate in mention order.")
        tokens = self.tokenizer(
            texts, padding=True, truncation=True, max_length=max_length, return_tensors="pt",
        )
        shape = (num_mentions, num_candidates, -1)
        return tokens["input_ids"].reshape(shape), tokens["attention_mask"].reshape(shape)


backbone_name = "FacebookAI/xlm-roberta-base"
tokenizer = LinkingTokenizer(backbone_name)


## 1. Mention detection

Encode one sentence and pool two mention spans. Gold BIO labels bypass the untrained classifier; offsets are start-inclusive and end-exclusive.


In [ ]:
class MentionDetection(nn.Module):
    """Encode, detect, decode, and pool two mentions in one input sequence."""
    def __init__(self, config, num_labels=3):
        super().__init__()
        self.transformer = XLMRobertaModel(config, add_pooling_layer=False)
        self.classifier = nn.Linear(config.hidden_size, num_labels)

    def forward(self, token_ids, mask_ids, text, offsets, bio_labels=None):
        token_vectors = self.transformer(input_ids=token_ids, attention_mask=mask_ids).last_hidden_state
        if bio_labels is None:
            bio_labels = self.classifier(token_vectors).argmax(-1)
        else:
            # Supplied labels bypass prediction; this does not train the classifier.
            if bio_labels.shape != token_ids.shape[1:] or bio_labels.dtype != torch.long:
                raise ValueError("bio_labels must be a 1D integer tensor with one label per token.")
            if not ((bio_labels >= 0) & (bio_labels <= 2)).all():
                raise ValueError("BIO labels must be 0=O, 1=B, or 2=I.")
            bio_labels = bio_labels.to(token_ids.device).unsqueeze(0)
        membership, mention_texts, mention_spans = self.decode(bio_labels, mask_ids, text, offsets)
        weights = membership.to(token_vectors.dtype)
        mention_vectors = weights @ token_vectors[0] / weights.sum(dim=1, keepdim=True)
        return mention_vectors, mention_texts, mention_spans

    def decode(self, bio_labels, mask_ids, text, offsets):
        if bio_labels.size(0) != 1:
            raise ValueError("This smoke test expects one input sequence.")
        content = mask_ids[0].bool() & (offsets[:, 1] > offsets[:, 0])
        labels = bio_labels[0].masked_fill(~content, 0)
        starts = labels.eq(1).nonzero().flatten()
        if starts.numel() != 2:
            raise ValueError("This smoke test expects exactly two B/I mentions.")
        previous = torch.cat((labels.new_zeros(1), labels[:-1]))
        if (labels.eq(2) & previous.eq(0)).any():
            raise ValueError("An I label must follow B or I.")

        # One mask per mention; B starts a group and I continues it.
        group_ids = labels.eq(1).cumsum(0) - 1
        membership = group_ids.unsqueeze(0).eq(torch.arange(2, device=labels.device).unsqueeze(1)) & labels.ne(0)
        positions = torch.arange(labels.numel(), device=labels.device).expand(2, -1)
        ends = positions.masked_fill(~membership, -1).max(dim=1).values
        mention_spans = torch.stack((offsets[starts, 0], offsets[ends, 1]), dim=1).tolist()
        mention_texts = [text[mention_spans[0][0]:mention_spans[0][1]],
                         text[mention_spans[1][0]:mention_spans[1][1]]]
        return membership, mention_texts, mention_spans

# Standard XLM-R base configuration, unchanged; weights are randomly initialized.
xlmr_config = XLMRobertaConfig.from_pretrained(backbone_name)
mention_detection = MentionDetection(xlmr_config).eval()


text = "Hawaii and Paris are places."
inputs = tokenizer.encode_text(text)
# Gold labels make this smoke test deterministic without training the model.
smoke_bio_labels = torch.tensor([0, 1, 0, 1, 0, 0, 0, 0])  # O, B, O, B, O, O, O, O.

with torch.no_grad():
    mention_vectors, mention_texts, mention_spans = mention_detection(**inputs, text=text, bio_labels=smoke_bio_labels)
print("Mention vectors:", mention_vectors.shape)
mention_table = pd.DataFrame(mention_vectors.detach().numpy()).add_prefix("feature_")
mention_table.insert(0, "end", [mention_spans[0][1], mention_spans[1][1]])
mention_table.insert(0, "start", [mention_spans[0][0], mention_spans[1][0]])
mention_table.insert(0, "mention", mention_texts)
display(mention_table.iloc[:, :11])


## 2. Entity typing: one probability per type

Predict categories from the mention vectors. The shared `type_names` order also defines the KB type-vector columns used by the next step.

In [ ]:
type_names = ["place", "state", "island"]

class EntityTyping(nn.Module):
    """Mention vectors -> independent probabilities for each type."""
    def __init__(self, num_types, hidden_size=8):
        super().__init__()
        self.classifier = nn.Linear(hidden_size, num_types)

    def forward(self, mention_vectors):
        return self.classifier(mention_vectors).sigmoid()

entity_typing = EntityTyping(num_types=len(type_names), hidden_size=mention_vectors.shape[-1]).eval()

with torch.no_grad():
    type_probabilities = entity_typing(mention_vectors)

type_table = pd.DataFrame(type_probabilities.detach().numpy(), columns=type_names)
type_table.insert(0, "mention", mention_texts)
display(type_table)


## 3. Candidate retrieval

Match names exactly and select two candidates per mention by prior. Keep mention order and original prior values. Missing or insufficient candidates raise an error; every returned slot is valid, including candidates with zero prior.


In [ ]:
# Each row is a possible entity for a stored name/alias.
# Priors are invented preferences, not measurements or model predictions.
kb = pd.DataFrame({
    "mention": ["Hawaii", "Hawaii", "Paris", "Paris"],
    "entity_id": ["Q782", "Q68740", "Q90", "Q830149"],
    "label": ["Hawaii", "Hawaii", "Paris", "Paris"],
    "description": ["state of the United States of America", "largest of the Hawaiian islands",
                    "capital of France", "city in Texas"],
    "prior": [0.85, 0.15, 0.90, 0.10],
    "place": [1., 1., 1., 1.],
    "state": [1., 0., 0., 0.],
    "island": [0., 1., 0., 0.],
})
display(kb)


In [ ]:
class CandidateRetriever:
    """Retrieve the highest-prior candidates for each mention, preserving mention order."""
    def __init__(self, kb: pd.DataFrame, type_names: list[str], num_candidates: int = 2):
        if not isinstance(num_candidates, int) or isinstance(num_candidates, bool) or num_candidates < 1:
            raise ValueError("num_candidates must be a positive integer.")
        self.kb = kb
        self.type_names = type_names
        self.num_candidates = num_candidates

    def __call__(self, mention_texts: list[str]) -> tuple[
        list[list[str]], pd.DataFrame, torch.Tensor, torch.Tensor
    ]:
        """Return IDs [M,C], records [M*C rows], known types [M,C,T], priors [M,C]."""
        num_mentions = len(mention_texts)
        if num_mentions == 0:
            raise ValueError("Provide at least one mention for this retrieval demo.")

        mention_rows = pd.DataFrame({"mention_index": range(num_mentions), "mention": mention_texts})
        records = mention_rows.merge(self.kb, on="mention", how="left", sort=False)
        if records["entity_id"].isna().any():
            raise ValueError("A mention has no candidates in the mock KB.")
        if records.groupby("mention_index").size().min() < self.num_candidates:
            raise ValueError("A mention has fewer KB candidates than requested; reduce num_candidates.")

        # Rank candidates within each mention.
        records = records.sort_values(["mention_index", "prior"], ascending=[True, False], kind="stable")
        records = records.groupby("mention_index", sort=False).head(self.num_candidates).reset_index(drop=True)

        candidate_ids = records["entity_id"].to_numpy().reshape(num_mentions, self.num_candidates).tolist()
        known_types = torch.tensor(records[self.type_names].to_numpy(), dtype=torch.float32)
        known_types = known_types.reshape(num_mentions, self.num_candidates, len(self.type_names))
        priors = torch.tensor(records["prior"].to_numpy(), dtype=torch.float32)
        priors = priors.reshape(num_mentions, self.num_candidates)
        return candidate_ids, records, known_types, priors


retriever = CandidateRetriever(kb=kb, type_names=type_names, num_candidates=2)
candidate_ids, candidate_records, known_types, priors = retriever(mention_texts)
num_mentions, num_candidates = priors.shape
# The retriever returns only real candidates, with no padded slots.
candidate_mask = torch.ones_like(priors, dtype=torch.bool)

display(candidate_records)


## 4. Description encoder: text becomes candidate vectors

Tokenize each retrieved entity's name + description with the same tokenizer used for the input sentence, then encode it. Padding and attention masks are created automatically.

In [ ]:
description_texts = (candidate_records["label"] + " " + candidate_records["description"]).tolist()
description_ids, description_mask = tokenizer.encode_descriptions(
    description_texts, num_mentions, num_candidates, max_length=32,
)


In [ ]:
class DescriptionEncoder(nn.Module):
    """Input [mentions, candidates, tokens]; output [mentions, candidates, features]."""
    def __init__(self, vocabulary_size, hidden_size=8, description_size=4):
        super().__init__()
        self.embedding = nn.Embedding(vocabulary_size, hidden_size, padding_idx=1)
        self.projection = nn.Linear(hidden_size, description_size)

    def forward(self, description_ids, description_mask):
        token_vectors = self.embedding(description_ids)
        token_mask = description_mask.bool().unsqueeze(-1)

        # Sum real-token vectors, then divide by their count.
        summed_vectors = (token_vectors * token_mask).sum(dim=-2)
        token_counts = token_mask.sum(dim=-2).clamp_min(1)
        mean_vectors = summed_vectors / token_counts
        return self.projection(mean_vectors)


description_encoder = DescriptionEncoder(
    vocabulary_size=xlmr_config.vocab_size, hidden_size=8, description_size=4,
).eval()

with torch.no_grad():
    description_vectors = description_encoder(description_ids, description_mask)
print("Description vectors:", description_vectors.shape)

description_table = candidate_records[["mention", "entity_id"]].reset_index(drop=True)
description_table["description_input"] = description_texts
vector_rows = description_vectors.detach().flatten(0, 1).numpy()
vector_table = pd.DataFrame(vector_rows).add_prefix("feature_")
description_table = pd.concat([description_table, vector_table], axis=1)
display(description_table)


## 5. Description matching: one probability per candidate, plus NIL

In [ ]:
class DescriptionMatching(nn.Module):
    """Project mention vectors; compare with candidate description vectors."""
    def __init__(self, hidden_size=8, description_size=4):
        super().__init__()
        self.mention_projection = nn.Linear(hidden_size, description_size)

    def forward(self, mention_vectors, description_vectors, candidate_mask):
        projected = self.mention_projection(mention_vectors)
        similarities = (description_vectors * projected.unsqueeze(1)).sum(-1)
        similarities = similarities.masked_fill(~candidate_mask, -1e8)
        none_score = similarities.new_zeros((similarities.size(0), 1))
        return torch.cat((similarities, none_score), dim=1).softmax(-1)

description_matching = DescriptionMatching(
    hidden_size=mention_vectors.shape[-1],
    description_size=description_vectors.shape[-1],
).eval()

with torch.no_grad():
    description_probabilities = description_matching(mention_vectors, description_vectors, candidate_mask)

score_entity_ids = candidate_ids[0] + ["NIL"] + candidate_ids[1] + ["NIL"]
score_mentions = [mention_texts[0]] * (num_candidates + 1) + [mention_texts[1]] * (num_candidates + 1)
match_table = pd.DataFrame({"mention": score_mentions, "entity_id": score_entity_ids,
                            "description_match": description_probabilities.detach().flatten().tolist()})
display(match_table)


## 6. Entity disambiguation

Score each candidate using per-type agreement, prior, type distance and description probability. Append a fixed NIL score of zero.


In [ ]:
class EntityDisambiguation(nn.Module):
    """Upstream scoring features with an explicit candidate validity mask."""
    def __init__(self, num_types):
        super().__init__()
        self.classifier = nn.Linear(num_types + 3, 1)

    def candidate_features(self, type_probabilities, priors,
                           known_types, description_probabilities):
        predicted_types = type_probabilities.unsqueeze(1)
        type_agreement = known_types * predicted_types
        type_distance = torch.linalg.vector_norm(known_types - predicted_types, dim=-1, keepdim=True)
        return torch.cat((type_agreement, priors.unsqueeze(-1),
                          type_distance, description_probabilities[:, :-1].unsqueeze(-1)), dim=-1)

    def forward(self, type_probabilities, priors, known_types,
                description_probabilities, candidate_mask):
        features = self.candidate_features(type_probabilities, priors,
                                           known_types, description_probabilities)
        scores = self.classifier(features).squeeze(-1)
        scores = scores.masked_fill(~candidate_mask, -1e8)
        none_score = scores.new_zeros((scores.size(0), 1))
        return torch.cat((scores, none_score), dim=1)

entity_disambiguation = EntityDisambiguation(num_types=len(type_names)).eval()

features = entity_disambiguation.candidate_features(type_probabilities, priors, known_types, description_probabilities)

evidence_table = pd.DataFrame(features.detach().reshape(-1, features.size(-1)).numpy(), columns=["place_agreement", "state_agreement", "island_agreement", "prior", "type_distance", "description_match"])
evidence_table.insert(0, "entity_id", candidate_records["entity_id"].tolist())
evidence_table.insert(0, "mention", candidate_records["mention"].tolist())
display(evidence_table)

with torch.no_grad():
    entity_scores = entity_disambiguation(
        type_probabilities=type_probabilities,
        priors=priors,
        known_types=known_types,
        description_probabilities=description_probabilities,
        candidate_mask=candidate_mask,
    )
assert entity_scores.shape == (num_mentions, num_candidates + 1)


## 7. Select the highest score; untrained scores have no factual meaning

In [ ]:
winning_columns = entity_scores.argmax(dim=1).tolist()
first_id = None if winning_columns[0] == num_candidates else candidate_ids[0][winning_columns[0]]
second_id = None if winning_columns[1] == num_candidates else candidate_ids[1][winning_columns[1]]
results = pd.DataFrame({
    "mention": mention_texts,
    "start": [mention_spans[0][0], mention_spans[1][0]],
    "end": [mention_spans[0][1], mention_spans[1][1]],
    "entity_id": [first_id, second_id],
    "untrained_demo": True,
})
score_table = pd.DataFrame({"mention": score_mentions, "entity_id": score_entity_ids,
                            "score": entity_scores.detach().flatten().tolist()})
score_table["selected"] = torch.arange(num_candidates + 1).unsqueeze(0).eq(entity_scores.argmax(dim=1, keepdim=True)).flatten().tolist()
display(score_table)
display(results)


## Weaknesses

| Weakness | What happens | Fix |
|---|---|---|
| Untrained weights | Scores do not establish the correct entity | Use the pretrained pipeline for real linking |
| Gold BIO labels | The demo checks decoding and pooling, not detection accuracy | Omit `bio_labels` to use classifier predictions after training |
| Fixed example | Gold BIO labels and mock candidates cover only Hawaii/Paris | Supply aligned labels and a KB for other inputs |
| Simplified description encoder | Uses mean embeddings instead of ReFinED's transformer | Use the pretrained description encoder or cached vectors |
